# Wearable-Only Baseline Machine Learning Models

## Objective

This notebook implements and evaluates baseline machine learning models using the participant-level wearable dataset for the AI-Assisted Screening of Parkinson's Disease project.

The analysis follows the shared modeling framework to ensure a consistent approach to data preprocessing, model training, evaluation, and result reporting across all data modalities. The wearable dataset is evaluated independently to establish a baseline for comparison with other modalities and future multimodal models.

### Baseline Models

- Logistic Regression
- Dummy Classifier
- Random Forest

In [5]:
# =============================================================================
# 1. IMPORT LIBRARIES
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from src.modeling.baseline import get_dummy_classifier
from src.modeling.models import (
    get_logistic_regression,
    get_random_forest
)

print("Basic libraries imported successfully.")

Basic libraries imported successfully.


In [6]:
# =============================================================================
# 2. SET PROJECT ROOT
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Users\kadih\AI-Assisted-Screening-of-Parkinson-s-Disease


In [7]:
# =============================================================================
# 3. IMPORT SHARED MODELING FRAMEWORK
# =============================================================================

from src.modeling import config
from src.modeling import data_loader
from src.modeling import split_validation
from src.modeling import preprocessing
from src.modeling import baseline
from src.modeling import models
from src.modeling import workflow
from src.modeling import evaluation
from src.modeling import outputs

print("Shared modeling framework imported successfully.")

Shared modeling framework imported successfully.


In [8]:
# =============================================================================
# 4. LOCATE PROCESSED DATA
# =============================================================================

DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Processed data directory:")
print(DATA_DIR)

print("\nAvailable files:")

for file in sorted(DATA_DIR.iterdir()):
    print(file.name)

Processed data directory:
c:\Users\kadih\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed

Available files:
.gitkeep
constant_features.csv
data_integration_validation_summary.csv
demographics_clean.csv
demographics_qa_summary.csv
feature_quality_report.csv
feature_quality_summary.csv
integrated_participant_dataset.csv
invalid_features.csv
low_variance_features.csv
movement_metadata.csv
participant_split.csv
patients.csv
preprocessed_signals
questionnaire.csv
questionnaire_cleaned.csv
test_metadata.csv
test_participant_dataset.csv
time_domain_features.csv
train_metadata.csv
train_participant_dataset.csv
validation_metadata.csv
validation_participant_dataset.csv
wearable_features.csv


In [9]:
# =============================================================================
# 5. LOAD PARTICIPANT-LEVEL WEARABLE DATASET
# =============================================================================

wearable_path = DATA_DIR / "wearable_features.csv"

if not wearable_path.exists():
    raise FileNotFoundError(
        f"Participant-level wearable dataset not found: {wearable_path}"
    )

wearable = pd.read_csv(wearable_path)

print("Participant-level wearable dataset loaded successfully.")
print("Shape:", wearable.shape)

display(wearable.head())

Participant-level wearable dataset loaded successfully.
Shape: (469, 183)


,patient_id,time_Accelerometer_X_Mean,time_Accelerometer_X_Median,time_Accelerometer_X_Std,time_Accelerometer_X_Min,time_Accelerometer_X_Max,time_Accelerometer_X_Range,time_Accelerometer_X_IQR,time_Accelerometer_X_RMS,time_Accelerometer_X_Energy,...,symmetry_Acc_Magnitude_Energy_difference,symmetry_Gyro_Magnitude_Mean_difference,symmetry_Gyro_Magnitude_Median_difference,symmetry_Gyro_Magnitude_Std_difference,symmetry_Gyro_Magnitude_Min_difference,symmetry_Gyro_Magnitude_Max_difference,symmetry_Gyro_Magnitude_Range_difference,symmetry_Gyro_Magnitude_IQR_difference,symmetry_Gyro_Magnitude_RMS_difference,symmetry_Gyro_Magnitude_Energy_difference
0,1,-0.000155,0.004059,0.074081,-0.513615,0.233092,0.746707,0.059540,0.074046,14.549401,...,1.631548,-0.126805,-0.085487,-0.123576,-0.007836,-0.317006,-0.309170,-0.028445,-0.179823,-943.554969
1,2,0.000043,-0.000302,0.062226,-0.365569,0.382510,0.748079,0.057380,0.062207,6.704495,...,-11.633377,-0.064486,-0.106319,0.023457,-0.004497,-0.520227,-0.515729,-0.031305,-0.028431,202.427722
2,3,-0.000092,-0.000014,0.089180,-0.364048,0.331769,0.695817,0.094730,0.089143,20.022262,...,0.987351,0.068282,0.084292,0.060849,0.017704,0.438968,0.421265,0.056464,0.092100,925.543812
3,4,-0.000061,-0.000984,0.035592,-0.177022,0.220743,0.397765,0.024906,0.035588,4.321860,...,-2.504678,0.014927,0.036575,0.000743,0.001043,0.170965,0.169922,0.052439,0.012388,-69.792417
4,5,-0.000042,-0.000507,0.060632,-0.213993,0.233428,0.447422,0.075401,0.060610,6.021132,...,-39.516070,-0.404767,-0.398273,-0.190974,-0.010912,-1.140813,-1.129902,-0.373412,-0.428032,-833.061251


In [10]:
# =============================================================================
# 6. LOAD PARTICIPANT SPLIT
# =============================================================================

split_path = DATA_DIR / "participant_split.csv"

participant_split = pd.read_csv(split_path)

print("Participant split loaded successfully.")

print("Shape:", participant_split.shape)

display(participant_split.head())

Participant split loaded successfully.
Shape: (469, 3)


,patient_id,label,split
0,1,0,train
1,2,2,train
2,3,0,validation
3,4,1,validation
4,5,1,train


In [11]:
# =============================================================================
# 7. INSPECT DATASET COLUMNS
# =============================================================================

print("Wearable dataset columns:")
print(wearable.columns.tolist())

print("\nParticipant split columns:")
print(participant_split.columns.tolist())

Wearable dataset columns:
['patient_id', 'time_Accelerometer_X_Mean', 'time_Accelerometer_X_Median', 'time_Accelerometer_X_Std', 'time_Accelerometer_X_Min', 'time_Accelerometer_X_Max', 'time_Accelerometer_X_Range', 'time_Accelerometer_X_IQR', 'time_Accelerometer_X_RMS', 'time_Accelerometer_X_Energy', 'time_Accelerometer_Y_Mean', 'time_Accelerometer_Y_Median', 'time_Accelerometer_Y_Std', 'time_Accelerometer_Y_Min', 'time_Accelerometer_Y_Max', 'time_Accelerometer_Y_Range', 'time_Accelerometer_Y_IQR', 'time_Accelerometer_Y_RMS', 'time_Accelerometer_Y_Energy', 'time_Accelerometer_Z_Mean', 'time_Accelerometer_Z_Median', 'time_Accelerometer_Z_Std', 'time_Accelerometer_Z_Min', 'time_Accelerometer_Z_Max', 'time_Accelerometer_Z_Range', 'time_Accelerometer_Z_IQR', 'time_Accelerometer_Z_RMS', 'time_Accelerometer_Z_Energy', 'time_Gyroscope_X_Mean', 'time_Gyroscope_X_Median', 'time_Gyroscope_X_Std', 'time_Gyroscope_X_Min', 'time_Gyroscope_X_Max', 'time_Gyroscope_X_Range', 'time_Gyroscope_X_IQR', 't

In [12]:
# =============================================================================
# 8. VERIFY PARTICIPANT-LEVEL STRUCTURE
# =============================================================================

ID_COLUMN = "patient_id"

print("Total rows:", len(wearable))
print("Unique participants:", wearable[ID_COLUMN].nunique())

duplicate_ids = wearable[ID_COLUMN].duplicated().sum()

print("Duplicate participant IDs:", duplicate_ids)

assert duplicate_ids == 0, (
    "Duplicate participant IDs detected in the wearable dataset."
)

print("✓ One row per participant confirmed.")

Total rows: 469
Unique participants: 469
Duplicate participant IDs: 0
✓ One row per participant confirmed.


In [13]:
# =============================================================================
# 9. VERIFY PARTICIPANT SPLIT
# =============================================================================

TARGET_COLUMN = "label"
SPLIT_COLUMN = "split"

print("Participants by split:")

display(
    participant_split.groupby(SPLIT_COLUMN)[ID_COLUMN]
    .nunique()
)

print("\nClass distribution:")

display(
    participant_split[TARGET_COLUMN]
    .value_counts(dropna=False)
)

Participants by split:


split
test           71
train         328
validation     70
Name: patient_id, dtype: int64


Class distribution:


label
1    276
2    114
0     79
Name: count, dtype: int64

In [14]:
# =============================================================================
# 10. MERGE WEARABLE FEATURES WITH LABELS AND SPLITS
# =============================================================================

wearable_model_data = wearable.merge(
    participant_split[
        [
            ID_COLUMN,
            TARGET_COLUMN,
            SPLIT_COLUMN
        ]
    ],
    on=ID_COLUMN,
    how="inner"
)

print("Final wearable modeling dataset:")
print(wearable_model_data.shape)

display(wearable_model_data.head())

Final wearable modeling dataset:
(469, 185)


,patient_id,time_Accelerometer_X_Mean,time_Accelerometer_X_Median,time_Accelerometer_X_Std,time_Accelerometer_X_Min,time_Accelerometer_X_Max,time_Accelerometer_X_Range,time_Accelerometer_X_IQR,time_Accelerometer_X_RMS,time_Accelerometer_X_Energy,...,symmetry_Gyro_Magnitude_Median_difference,symmetry_Gyro_Magnitude_Std_difference,symmetry_Gyro_Magnitude_Min_difference,symmetry_Gyro_Magnitude_Max_difference,symmetry_Gyro_Magnitude_Range_difference,symmetry_Gyro_Magnitude_IQR_difference,symmetry_Gyro_Magnitude_RMS_difference,symmetry_Gyro_Magnitude_Energy_difference,label,split
0,1,-0.000155,0.004059,0.074081,-0.513615,0.233092,0.746707,0.059540,0.074046,14.549401,...,-0.085487,-0.123576,-0.007836,-0.317006,-0.309170,-0.028445,-0.179823,-943.554969,0,train
1,2,0.000043,-0.000302,0.062226,-0.365569,0.382510,0.748079,0.057380,0.062207,6.704495,...,-0.106319,0.023457,-0.004497,-0.520227,-0.515729,-0.031305,-0.028431,202.427722,2,train
2,3,-0.000092,-0.000014,0.089180,-0.364048,0.331769,0.695817,0.094730,0.089143,20.022262,...,0.084292,0.060849,0.017704,0.438968,0.421265,0.056464,0.092100,925.543812,0,validation
3,4,-0.000061,-0.000984,0.035592,-0.177022,0.220743,0.397765,0.024906,0.035588,4.321860,...,0.036575,0.000743,0.001043,0.170965,0.169922,0.052439,0.012388,-69.792417,1,validation
4,5,-0.000042,-0.000507,0.060632,-0.213993,0.233428,0.447422,0.075401,0.060610,6.021132,...,-0.398273,-0.190974,-0.010912,-1.140813,-1.129902,-0.373412,-0.428032,-833.061251,1,train


In [15]:
# =============================================================================
# 11. VALIDATE FINAL MODELING DATASET
# =============================================================================

print("Participants:", wearable_model_data[ID_COLUMN].nunique())

print("\nClass distribution:")
display(
    wearable_model_data[TARGET_COLUMN]
    .value_counts(dropna=False)
)

print("\nSplit distribution:")
display(
    wearable_model_data[SPLIT_COLUMN]
    .value_counts(dropna=False)
)

print("\nMissing labels:")
print(
    wearable_model_data[TARGET_COLUMN].isna().sum()
)

assert wearable_model_data[TARGET_COLUMN].notna().all(), (
    "Missing target labels detected."
)

print("\n✓ Final wearable modeling dataset validated.")

Participants: 469

Class distribution:


label
1    276
2    114
0     79
Name: count, dtype: int64


Split distribution:


split
train         328
test           71
validation     70
Name: count, dtype: int64


Missing labels:
0

✓ Final wearable modeling dataset validated.


In [16]:
# =============================================================================
# 12. SELECT WEARABLE FEATURES
# =============================================================================

EXCLUDED_COLUMNS = {
    ID_COLUMN,
    TARGET_COLUMN,
    SPLIT_COLUMN
}

wearable_feature_columns = [
    column
    for column in wearable_model_data.columns
    if column not in EXCLUDED_COLUMNS
]

print("Total wearable features:", len(wearable_feature_columns))

Total wearable features: 182


In [17]:
# =============================================================================
# 13. IDENTIFY NUMERICAL WEARABLE FEATURES
# =============================================================================

numeric_wearable_features = (
    wearable_model_data[wearable_feature_columns]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

non_numeric_features = [
    column
    for column in wearable_feature_columns
    if column not in numeric_wearable_features
]

print("Total wearable features:", len(wearable_feature_columns))
print("Numerical wearable features:", len(numeric_wearable_features))
print("Non-numerical features:", len(non_numeric_features))

if non_numeric_features:
    print("\nNon-numerical columns:")
    print(non_numeric_features)

Total wearable features: 182
Numerical wearable features: 182
Non-numerical features: 0


In [18]:
# =============================================================================
# 14. CHECK MISSING WEARABLE FEATURES
# =============================================================================

missing_features = (
    wearable_model_data[numeric_wearable_features]
    .isna()
    .sum()
)

missing_features = (
    missing_features[missing_features > 0]
    .sort_values(ascending=False)
)

if missing_features.empty:
    print("✓ No missing wearable feature values found.")
else:
    print("Missing wearable feature values:")
    display(missing_features)

✓ No missing wearable feature values found.


In [19]:
# =============================================================================
# 15. VERIFY PARTICIPANT-LEVEL DATA SPLITS
# =============================================================================

train_ids = set(
    wearable_model_data.loc[
        wearable_model_data[SPLIT_COLUMN].str.lower() == "train",
        ID_COLUMN
    ]
)

validation_ids = set(
    wearable_model_data.loc[
        wearable_model_data[SPLIT_COLUMN].str.lower().isin(
            ["validation", "val"]
        ),
        ID_COLUMN
    ]
)

test_ids = set(
    wearable_model_data.loc[
        wearable_model_data[SPLIT_COLUMN].str.lower() == "test",
        ID_COLUMN
    ]
)

print("Train participants:", len(train_ids))
print("Validation participants:", len(validation_ids))
print("Test participants:", len(test_ids))

assert len(train_ids & validation_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0

print("\n✓ No participant leakage detected.")

Train participants: 328
Validation participants: 70
Test participants: 71

✓ No participant leakage detected.


In [20]:
# =============================================================================
# 16. CONFIRM NO PARTICIPANT LEAKAGE
# =============================================================================

assert len(train_ids & validation_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0

print(
    "✓ No participant appears in multiple dataset splits."
)

✓ No participant appears in multiple dataset splits.


In [21]:
# =============================================================================
# 17. CHECK WEARABLE FEATURE TYPES
# =============================================================================

feature_types = (
    wearable_model_data[
        wearable_feature_columns
    ]
    .dtypes
)

display(feature_types.value_counts())

float64    182
Name: count, dtype: int64

In [22]:
# =============================================================================
# 18. KEEP NUMERICAL WEARABLE FEATURES
# =============================================================================

numeric_wearable_features = (
    wearable_model_data[
        wearable_feature_columns
    ]
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

print(
    "Total wearable features:",
    len(wearable_feature_columns)
)

print(
    "Numerical wearable features:",
    len(numeric_wearable_features)
)

non_numeric_features = [
    column
    for column in wearable_feature_columns
    if column not in numeric_wearable_features
]

print(
    "Non-numerical features:",
    len(non_numeric_features)
)

if non_numeric_features:
    print("\nNon-numerical columns:")
    print(non_numeric_features)

Total wearable features: 182
Numerical wearable features: 182
Non-numerical features: 0


In [23]:
# =============================================================================
# 19. FINAL WEARABLE FEATURE MATRIX
# =============================================================================

X_wearable = wearable_model_data[
    numeric_wearable_features
].copy()

y = wearable_model_data[
    TARGET_COLUMN
].copy()

groups = wearable_model_data[
    ID_COLUMN
].copy()

print("X shape:", X_wearable.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)

print("\nNumber of features:")
print(X_wearable.shape[1])

print("\nNumber of participants:")
print(X_wearable.shape[0])

X shape: (469, 182)
y shape: (469,)
groups shape: (469,)

Number of features:
182

Number of participants:
469


In [24]:
# =============================================================================
# 20. FINAL SANITY CHECK BEFORE MODELING
# =============================================================================

print("=" * 70)
print("WEARABLE DATA READY FOR SHARED MODELING FRAMEWORK")
print("=" * 70)

print(
    "Participants:",
    X_wearable.shape[0]
)

print(
    "Wearable features:",
    X_wearable.shape[1]
)

print(
    "Classes:",
    sorted(y.dropna().unique())
)

print("\nClass counts:")
display(y.value_counts())

print("\nSplit counts:")
display(
    wearable_model_data[
        SPLIT_COLUMN
    ].value_counts()
)

print("\n✓ Data preparation completed.")

WEARABLE DATA READY FOR SHARED MODELING FRAMEWORK
Participants: 469
Wearable features: 182
Classes: [np.int64(0), np.int64(1), np.int64(2)]

Class counts:


label
1    276
2    114
0     79
Name: count, dtype: int64


Split counts:


split
train         328
test           71
validation     70
Name: count, dtype: int64


✓ Data preparation completed.


## Wearable-Only Modeling

The wearable-only analysis uses participant-level wearable features derived from smartwatch movement recordings. The dataset is divided into training, validation, and independent test sets at the participant level to prevent participant overlap across datasets.

In [25]:
# =============================================================================
# 21. CREATE TRAIN, VALIDATION, AND TEST DATASETS
# =============================================================================

train_df = wearable_model_data[
    wearable_model_data[SPLIT_COLUMN].str.lower() == "train"
].copy()

validation_df = wearable_model_data[
    wearable_model_data[SPLIT_COLUMN].str.lower().isin(["validation", "val"])
].copy()

test_df = wearable_model_data[
    wearable_model_data[SPLIT_COLUMN].str.lower() == "test"
].copy()

print("Participant-level datasets created successfully.")

print("\nDataset shapes:")
print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Testing:", test_df.shape)

print("\nParticipant counts:")
print("Training:", train_df[ID_COLUMN].nunique())
print("Validation:", validation_df[ID_COLUMN].nunique())
print("Testing:", test_df[ID_COLUMN].nunique())

Participant-level datasets created successfully.

Dataset shapes:
Training: (328, 185)
Validation: (70, 185)
Testing: (71, 185)

Participant counts:
Training: 328
Validation: 70
Testing: 71


## Model Configuration

Three baseline classification models are evaluated using the wearable-only feature set: Dummy Classifier, Logistic Regression, and Random Forest. The models use the shared preprocessing and modeling framework established for the project.

In [26]:
# =============================================================================
# 22. DEFINE WEARABLE-ONLY MODELS
# =============================================================================

models = {
    "Dummy Classifier": get_dummy_classifier(),
    "Logistic Regression": get_logistic_regression(),
    "Random Forest": get_random_forest()
}

print("Wearable-only models:")

for model_name in models:
    print("-", model_name)

Wearable-only models:
- Dummy Classifier
- Logistic Regression
- Random Forest


## Model Training and Evaluation

The models are trained on the participant-level training dataset and evaluated using the validation and independent test datasets. Performance is assessed using accuracy, balanced accuracy, macro F1-score, macro precision, and macro recall.

In [27]:
# =============================================================================
# 23. RUN WEARABLE-ONLY MODELS
# =============================================================================

wearable_results = workflow.run_models(
    models=models,
    train_df=train_df,
    validation_df=validation_df,
    test_df=test_df
)

print("Wearable-only models completed successfully.")

print("\nModels evaluated:")
for model_name in wearable_results:
    print("-", model_name)

Wearable-only models completed successfully.

Models evaluated:
- Dummy Classifier
- Logistic Regression
- Random Forest


In [28]:
# =============================================================================
# 24. DISPLAY RESULTS FOR ALL WEARABLE-ONLY MODELS
# =============================================================================

for model_name, result in wearable_results.items():
    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("\nValidation metrics:")
    for metric, value in result["metrics"].items():
        print(f"{metric}: {value:.4f}")

    if "test_metrics" in result:
        print("\nTest metrics:")
        for metric, value in result["test_metrics"].items():
            print(f"{metric}: {value:.4f}")

Dummy Classifier

Validation metrics:
accuracy: 0.5857
balanced_accuracy: 0.3333
macro_f1: 0.2462
precision_macro: 0.1952
recall_macro: 0.3333

Test metrics:
accuracy: 0.5915
balanced_accuracy: 0.3333
macro_f1: 0.2478
precision_macro: 0.1972
recall_macro: 0.3333
Logistic Regression

Validation metrics:
accuracy: 0.5714
balanced_accuracy: 0.5546
macro_f1: 0.5385
precision_macro: 0.5330
recall_macro: 0.5546

Test metrics:
accuracy: 0.5211
balanced_accuracy: 0.4827
macro_f1: 0.4764
precision_macro: 0.4731
recall_macro: 0.4827
Random Forest

Validation metrics:
accuracy: 0.6000
balanced_accuracy: 0.5790
macro_f1: 0.5576
precision_macro: 0.5493
recall_macro: 0.5790

Test metrics:
accuracy: 0.6338
balanced_accuracy: 0.5894
macro_f1: 0.5805
precision_macro: 0.5772
recall_macro: 0.5894


In [29]:
# =============================================================================
# 25. DISPLAY CLASSIFICATION REPORTS
# =============================================================================

for model_name, result in wearable_results.items():

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("\nValidation Classification Report:")
    display(pd.DataFrame(result["classification_report"]).T)

    if "test_report" in result:
        print("\nTest Classification Report:")
        display(pd.DataFrame(result["test_report"]).T)

Dummy Classifier

Validation Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.0,0.585714,0.0,0.585714,0.195238,0.343061
recall,0.0,1.000000,0.0,0.585714,0.333333,0.585714
f1-score,0.0,0.738739,0.0,0.585714,0.246246,0.432690
support,12.0,41.000000,17.0,0.585714,70.000000,70.000000



Test Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.0,0.591549,0.0,0.591549,0.197183,0.349931
recall,0.0,1.000000,0.0,0.591549,0.333333,0.591549
f1-score,0.0,0.743363,0.0,0.591549,0.247788,0.439736
support,12.0,42.000000,17.0,0.591549,71.000000,71.000000


Logistic Regression

Validation Classification Report:


...

,0,1,2,accuracy,macro avg,weighted avg
precision,0.461538,0.657895,0.300000,0.521127,0.473144,0.539015
recall,0.500000,0.595238,0.352941,0.521127,0.482726,0.521127
f1-score,0.480000,0.625000,0.324324,0.521127,0.476441,0.528500
support,12.000000,42.000000,17.000000,0.521127,71.000000,71.000000


Random Forest

Validation Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.470588,0.710526,0.466667,0.6,0.549260,0.610171
recall,0.666667,0.658537,0.411765,0.6,0.578989,0.600000
f1-score,0.551724,0.683544,0.437500,0.6,0.557589,0.601193
support,12.000000,41.000000,17.000000,0.6,70.000000,70.000000



Test Classification Report:


,0,1,2,accuracy,macro avg,weighted avg
precision,0.466667,0.731707,0.533333,0.633803,0.577236,0.639414
recall,0.583333,0.714286,0.470588,0.633803,0.589402,0.633803
f1-score,0.518519,0.722892,0.500000,0.633803,0.580470,0.634981
support,12.000000,42.000000,17.000000,0.633803,71.000000,71.000000


## Confusion Matrix Analysis

Confusion matrices are examined to identify class-level prediction errors and determine which participant classes are most frequently confused by the wearable-only models.

In [30]:
# =============================================================================
# 26. CONFUSION MATRICES
# =============================================================================

for model_name, result in wearable_results.items():

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("\nValidation Confusion Matrix:")
    display(result["confusion_matrix"])

    if "test_confusion_matrix" in result:
        print("\nTest Confusion Matrix:")
        display(result["test_confusion_matrix"])

Dummy Classifier

Validation Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,41,0
True_Other,0,17,0



Test Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,42,0
True_Other,0,17,0


Logistic Regression

Validation Confusion Matrix:


...

,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,6,4,2
True_PD,5,25,12
True_Other,2,9,6


Random Forest

Validation Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,8,4,0
True_PD,6,27,8
True_Other,3,7,7



Test Confusion Matrix:


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,7,4,1
True_PD,6,30,6
True_Other,2,7,8


In [31]:
# =============================================================================
# 27. WEARABLE-ONLY MODEL COMPARISON
# =============================================================================

comparison_rows = []

for model_name, result in wearable_results.items():

    row = {
        "Model": model_name,
        "Validation Accuracy": result["metrics"]["accuracy"],
        "Validation Balanced Accuracy": result["metrics"]["balanced_accuracy"],
        "Validation Macro F1": result["metrics"]["macro_f1"],
        "Validation Macro Precision": result["metrics"]["precision_macro"],
        "Validation Macro Recall": result["metrics"]["recall_macro"],
    }

    if "test_metrics" in result:
        row.update({
            "Test Accuracy": result["test_metrics"]["accuracy"],
            "Test Balanced Accuracy": result["test_metrics"]["balanced_accuracy"],
            "Test Macro F1": result["test_metrics"]["macro_f1"],
            "Test Macro Precision": result["test_metrics"]["precision_macro"],
            "Test Macro Recall": result["test_metrics"]["recall_macro"],
        })

    comparison_rows.append(row)

wearable_comparison = pd.DataFrame(comparison_rows)

display(wearable_comparison.round(4))

,Model,Validation Accuracy,Validation Balanced Accuracy,Validation Macro F1,Validation Macro Precision,Validation Macro Recall,Test Accuracy,Test Balanced Accuracy,Test Macro F1,Test Macro Precision,Test Macro Recall
0,Dummy Classifier,0.5857,0.3333,0.2462,0.1952,0.3333,0.5915,0.3333,0.2478,0.1972,0.3333
1,Logistic Regression,0.5714,0.5546,0.5385,0.5330,0.5546,0.5211,0.4827,0.4764,0.4731,0.4827
2,Random Forest,0.6000,0.5790,0.5576,0.5493,0.5790,0.6338,0.5894,0.5805,0.5772,0.5894


In [33]:
# =============================================================================
# 28. SAVE WEARABLE-ONLY OUTPUTS
# =============================================================================

# Save model comparison table
outputs.save_model_comparison(
    wearable_comparison,
    "wearable_model_comparison.csv"
)

# Save metrics, classification reports, and confusion matrices
for model_name, result in wearable_results.items():

    # Create a safe filename from the model name
    model_file_name = model_name.lower().replace(" ", "_")

    # Save validation metrics
    outputs.save_metrics(
        result["metrics"],
        f"wearable_{model_file_name}_validation_metrics.csv"
    )

    # Save test metrics
    if "test_metrics" in result:
        outputs.save_metrics(
            result["test_metrics"],
            f"wearable_{model_file_name}_test_metrics.csv"
        )

    # Save validation classification report
    validation_report = pd.DataFrame(
        result["classification_report"]
    ).T

    outputs.save_classification_report(
        validation_report,
        f"wearable_{model_file_name}_validation_classification_report.csv"
    )

    # Save test classification report
    if "test_report" in result:
        test_report = pd.DataFrame(
            result["test_report"]
        ).T

        outputs.save_classification_report(
            test_report,
            f"wearable_{model_file_name}_test_classification_report.csv"
        )

    # Save validation confusion matrix
    outputs.save_confusion_matrix(
        result["confusion_matrix"],
        f"wearable_{model_file_name}_validation_confusion_matrix.csv"
    )

    # Save test confusion matrix
    if "test_confusion_matrix" in result:
        outputs.save_confusion_matrix(
            result["test_confusion_matrix"],
            f"wearable_{model_file_name}_test_confusion_matrix.csv"
        )

print("Wearable-only outputs saved successfully.")

Wearable-only outputs saved successfully.


## Summary

The wearable-only models showed different levels of classification performance across the three baseline approaches. The Dummy Classifier achieved a validation accuracy of 58.57%, with a balanced accuracy of 33.33% and a macro F1-score of 24.62%, providing a reference baseline for comparison. Logistic Regression achieved a validation accuracy of 57.14%, balanced accuracy of 55.46%, and macro F1-score of 53.85%. Random Forest performed best on the validation set, achieving an accuracy of 60.00%, balanced accuracy of 57.90%, and macro F1-score of 55.76%.

On the independent test set, Random Forest achieved the strongest overall performance, with an accuracy of 63.38%, balanced accuracy of 58.94%, and macro F1-score of 58.05%. The Dummy Classifier achieved 59.15% accuracy but substantially lower balanced accuracy (33.33%) and macro F1-score (24.78%), indicating that its accuracy was influenced by class imbalance. The confusion matrices also showed that the models classified the Parkinson's disease class more successfully than the Other movement-disorder class, which experienced greater misclassification.

Overall, Random Forest provided the strongest wearable-only baseline performance and will serve as a useful reference for comparison with the other feature modalities and the final multimodal model.